# Result Table Generator

Gurobi experiments following Liberti, Manca and Poirion (2023), Tables 3–6. Only the small smoke examples execute by default. See `docs/result-table-generator.md` and `docs/result-table-methodology.md` for usage and the source audit. These are Gurobi adaptations; published numerical results and timings are not expected to match exactly.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/result_table_generator.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.result_table_generator import result_table_generator

# The function defaults to ROOT.parent / 'thesis_results'.
# Keep this notebook's diagnostics together; override this path as needed.
OUTPUT_DIR = ROOT / 'results' / 'result_table_generator_smoke'
RUN_PUBLISHED = False


## Small QR smoke experiment

There are always 399 predictors. Using 450 records allows a nonzero original objective; very small record counts can have zero optimum and undefined objective ratios. Each row below averages two independent projections of the same instance.


In [ ]:
qr_smoke = result_table_generator(
    'Quantile Regression',
    epsilon_min=0.2, epsilon_max=0.5, epsilon_increment=0.3,
    row_min=450, row_max=450, row_increment=1,
    num_runs=2, master_seed=2026, experiment_name='qr-smoke',
    output_dir=OUTPUT_DIR,
)
display(qr_smoke.preview('QR'))
display(qr_smoke.counts['QR'])
print(qr_smoke.output_path)


## Small Diet smoke experiment

One call runs both TDP1 and TDP2, paired by projection seed. Success means the LP solves and retrieval completed; feasibility flags and errors are reported separately.


In [ ]:
diet_smoke = result_table_generator(
    'Diet Problem',
    epsilon_min=0.2, epsilon_max=0.5, epsilon_increment=0.3,
    row_min=80, row_max=80, row_increment=1,
    num_runs=2, master_seed=2026, experiment_name='diet-smoke',
    output_dir=OUTPUT_DIR,
)
for variant in ('TDP1', 'TDP2'):
    display(diet_smoke.preview(variant))
    display(diet_smoke.counts[variant])
display(diet_smoke.raw_trials[['epsilon', 'replication', 'formulation_variant',
                             'instance_seed', 'projection_seed', 'projection_digest',
                             'projected_status', 'retrieval_status', 'original_feasible']])


## Display saved plots and high-precision data

The workbook includes raw and averaged sheets, plus separate variant tables and counts. Excel seeds are stored as text to preserve all 64 bits.


In [ ]:
display(Image(filename=str(qr_smoke.files['QR_objectives.png'])))
display(Image(filename=str(diet_smoke.files['TDP1_feasibility.png'])))
display(Image(filename=str(diet_smoke.files['TDP2_size_cpu.png'])))
display(qr_smoke.averaged_results)
print('Workbook:', qr_smoke.files['results.xlsx'])
print('Metadata:', qr_smoke.files['metadata.json'])


## Table 5: Quantile Regression, epsilon 0.2–0.9

The following three cells require `RUN_PUBLISHED = True`. Five runs per size/epsilon pair; every original instance is solved once. Large experiments have not been executed as part of the initial smoke validation.


In [ ]:
if RUN_PUBLISHED:
    table5 = result_table_generator(
        'Quantile Regression', 0.2, 0.9, 0.1, 5000, 9000, 1000,
        num_runs=5, experiment_name='qr-table5', master_seed=2026,
    )
    display(table5.preview('QR'))
    display(table5.counts['QR'])


## Table 6: Quantile Regression, epsilon 0.10 and 0.15

These are Table 6's exact epsilon values; the upstream commented list also contains other values. The configuration hash keeps this sweep separate from Table 5.


In [ ]:
if RUN_PUBLISHED:
    table6 = result_table_generator(
        'Quantile Regression', 0.1, 0.15, 0.05, 5000, 9000, 1000,
        num_runs=5, experiment_name='qr-table6', master_seed=2026,
    )
    display(table6.preview('QR'))
    display(table6.counts['QR'])


## Tables 3–4: Diet Problem


In [ ]:
if RUN_PUBLISHED:
    diet_tables = result_table_generator(
        'Diet Problem', 0.2, 0.9, 0.1, 3000, 7000, 1000,
        num_runs=5, experiment_name='diet-tables3-4', master_seed=2026,
    )
    for variant in ('TDP1', 'TDP2'):
        display(diet_tables.preview(variant))
        display(diet_tables.counts[variant])


## Reproduce one saved replication

Use the recorded seeds and the same solver/library settings. Timings naturally vary; projected matrices should have matching digests. This calls the single-replication interface directly.


In [ ]:
from src.result_tables import quantile_regression as qr

trial = qr_smoke.raw_trials.iloc[0]
instance = qr.generate_instance(int(trial.row_size), int(trial.instance_seed))
original = qr.solve_original(instance)
replayed = qr.solve_one_replication(instance, float(trial.epsilon),
                                    int(trial.projection_seed), original)
assert replayed['projection_digest'] == trial.projection_digest
print({key: replayed[key] for key in ('f_original', 'f_projected', 'f_retrieved',
                                    'projected_status', 'retrieval_status')})
